In [ ]:
# One-time migration — adds surthost/surtpath columns to the cdxj DB (already applied on the server):
# !venv/bin/python add_surtkey_columns.py --db-path data/domains/12_transportation/cdxj.duckdb


In [ ]:
import sys
from pathlib import Path

# Make repo-root imports work whether the kernel cwd is the repo root or domain_analysis/
if not Path('domain_analysis.py').exists():
    sys.path.insert(0, '..')

import duckdb
import pandas as pd

from domain_analysis import (
    base_domain_sql,
    subdomain_sql,
    build_domain_summary,
    build_subdomain_breakdown,
)

pd.set_option('display.max_rows', 120)
pd.set_option('display.max_colwidth', 80)

# DB has been migrated with add_surtkey_columns.py:
#   surthost, surthost_seg_0..5, surtpath_1..5
_cdxj_candidates = [
    Path('data/12_transportation/cdxj.duckdb'),             # local layout (cwd = repo root)
    Path('../data/12_transportation/cdxj.duckdb'),          # local layout (cwd = domain_analysis/)
    Path('data/domains/12_transportation/cdxj.duckdb'),     # server layout (cwd = repo root)
    Path('../data/domains/12_transportation/cdxj.duckdb'),  # server layout (cwd = domain_analysis/)
]
CDXJ_DB = next((p for p in _cdxj_candidates if p.exists()), None)
if CDXJ_DB is None:
    raise FileNotFoundError(
        'cdxj.duckdb not found — the cdxj sections need the server DB. '
        'The Deduplicated URL Analysis section below only needs parquet.duckdb.')
transportation_con = duckdb.connect(str(CDXJ_DB), read_only=True)

from config import TARGET_DOMAINS  # substitution-safe (not a hardcoded list)

print(f'Connected to {CDXJ_DB}')

In [ ]:
transportation_con.sql("SELECT COUNT(*) FROM eot_captures").show()

In [ ]:
# Total captures per year
# NOTE: no 2012 data in the per-domain DB, so 2012 is missing here
transportation_con.sql("""
    SELECT crawl_year, COUNT(*) AS captures
    FROM eot_captures
    GROUP BY 1 ORDER BY 1
""").show()

In [ ]:
# host, year, count
transportation_con.sql("""
    SELECT host, crawl_year, COUNT(*) AS n
    FROM eot_captures
    GROUP BY 1, 2
    ORDER BY n DESC
""").show(max_rows=50)

### SURT host columns — materialized via `add_surtkey_columns.py`

The DB now carries `surthost`, `surthost_seg_0..5`, and `surtpath_1..5` as physical columns
parsed from the SURT key. We can query them directly instead of regex-extracting from `surtkey`.

**Heads up:** SURT canonicalization strips a leading `www`, so `www.transportation.gov` rows
have the same `surthost` as bare `transportation.gov` (`gov,transportation`) and their
`surthost_seg_2` is NULL.

In [ ]:
# Show the new columns populated against a few representative hosts
transportation_con.sql("""
    SELECT host, surthost,
           surthost_seg_0 AS tld,
           surthost_seg_1 AS reg_dom,
           surthost_seg_2 AS sub1,
           surthost_seg_3 AS sub2,
           surtpath_1, surtpath_2
    FROM eot_captures
    WHERE host IN ('www.transportation.gov', 'data.transportation.gov',
                   'www7.transportation.gov', 'datahub.transportation.gov')
    LIMIT 8
""").show()


### DomainDataSummary — high-level domain structure (SURT-derived)

In [ ]:
# Per-domain roll-up parsed from surtkey (no hardcoded host list).
domain_summary = build_domain_summary(transportation_con)
domain_summary


In [ ]:
# Full subdomain breakdown — uses materialized surthost_seg_2 column directly.
# Note: surthost_seg_2 is the innermost subdomain label after SURT canonicalization
# (so 'www.foo.gov' rows fold into NULL→'(bare/www)' alongside bare 'foo.gov').
subdomain_df = transportation_con.sql("""
    SELECT
        COALESCE(surthost_seg_2, '(bare/www)') AS subdomain,
        crawl_year,
        COUNT(*) AS n
    FROM eot_captures
    WHERE surtkey IS NOT NULL
    GROUP BY 1, 2
""").df()

subdomain_pivot = (subdomain_df
    .pivot_table(index='subdomain', columns='crawl_year', values='n', aggfunc='sum')
    .fillna(0).astype(int))
subdomain_pivot['total'] = subdomain_pivot.sum(axis=1)
subdomain_pivot = subdomain_pivot.sort_values('total', ascending=False).head(30)

total_subs = transportation_con.sql("""
    SELECT COUNT(DISTINCT COALESCE(surthost_seg_2, '(bare/www)'))
    FROM eot_captures WHERE surtkey IS NOT NULL
""").fetchone()[0]
print(f"Total unique top-level subdomain labels (surthost_seg_2): {total_subs:,}")
subdomain_pivot


In [ ]:
# Position 1 (surtpath_1) — ranked per year
seg1 = transportation_con.sql("""
    SELECT
        COALESCE(NULLIF(surtpath_1, ''), '(root)') AS seg1,
        crawl_year,
        COUNT(*) AS n
    FROM eot_captures
    GROUP BY 1, 2
""").df()

seg1_pivot = seg1.pivot(index='seg1', columns='crawl_year', values='n').fillna(0).astype(int)
seg1_pivot['total'] = seg1_pivot.sum(axis=1)
seg1_pivot = seg1_pivot.sort_values('total', ascending=False)
seg1_pivot

In [ ]:
seg1_pivot.head(50)

In [ ]:
seg1_pivot[seg1_pivot[['2004', '2008']].sum(axis=1) > 0][['2004', '2008', 'total']].sort_values('2004', ascending=False)


In [ ]:
seg1_pivot[seg1_pivot[['2016']].sum(axis=1) > 0][['2016', 'total']].sort_values('2016', ascending=False)


In [ ]:
# Position 2 (surtpath_2) — ranked per year
seg2 = transportation_con.sql("""
    SELECT
        COALESCE(NULLIF(surtpath_1, ''), '(root)') AS seg1,
        COALESCE(NULLIF(surtpath_2, ''), '(none)') AS seg2,
        crawl_year,
        COUNT(*) AS n
    FROM eot_captures
    GROUP BY 1, 2, 3
""").df()

seg2_pivot = seg2.pivot_table(index=['seg1', 'seg2'], columns='crawl_year', values='n',
                              aggfunc='sum').fillna(0).astype(int)
seg2_pivot['total'] = seg2_pivot.sum(axis=1)
seg2_pivot = seg2_pivot.sort_values('total', ascending=False)
seg2_pivot

In [ ]:
# Position 3 (surtpath_3) — ranked per year
seg3 = transportation_con.sql("""
    SELECT
        COALESCE(NULLIF(surtpath_1, ''), '(root)') AS seg1,
        COALESCE(NULLIF(surtpath_2, ''), '(none)') AS seg2,
        COALESCE(NULLIF(surtpath_3, ''), '(none)') AS seg3,
        crawl_year,
        COUNT(*) AS n
    FROM eot_captures
    GROUP BY 1, 2, 3, 4
""").df()

seg3_pivot = seg3.pivot_table(index=['seg1', 'seg2', 'seg3'], columns='crawl_year', values='n',
                              aggfunc='sum').fillna(0).astype(int)
seg3_pivot['total'] = seg3_pivot.sum(axis=1)
seg3_pivot = seg3_pivot.sort_values('total', ascending=False)
seg3_pivot

In [ ]:
# Position 4 (surtpath_4) — ranked per year
seg4 = transportation_con.sql("""
    SELECT
        COALESCE(NULLIF(surtpath_1, ''), '(root)') AS seg1,
        COALESCE(NULLIF(surtpath_2, ''), '(none)') AS seg2,
        COALESCE(NULLIF(surtpath_3, ''), '(none)') AS seg3,
        COALESCE(NULLIF(surtpath_4, ''), '(none)') AS seg4,
        crawl_year,
        COUNT(*) AS n
    FROM eot_captures
    GROUP BY 1, 2, 3, 4, 5
""").df()

seg4_pivot = seg4.pivot_table(index=['seg1', 'seg2', 'seg3', 'seg4'], columns='crawl_year', values='n',
                              aggfunc='sum').fillna(0).astype(int)
seg4_pivot['total'] = seg4_pivot.sum(axis=1)
seg4_pivot = seg4_pivot.sort_values('total', ascending=False)
seg4_pivot

In [ ]:
# Position 5 (surtpath_5) — ranked per year
seg5 = transportation_con.sql("""
    SELECT
        COALESCE(NULLIF(surtpath_1, ''), '(root)') AS seg1,
        COALESCE(NULLIF(surtpath_2, ''), '(none)') AS seg2,
        COALESCE(NULLIF(surtpath_3, ''), '(none)') AS seg3,
        COALESCE(NULLIF(surtpath_4, ''), '(none)') AS seg4,
        COALESCE(NULLIF(surtpath_5, ''), '(none)') AS seg5,
        crawl_year,
        COUNT(*) AS n
    FROM eot_captures
    GROUP BY 1, 2, 3, 4, 5, 6
""").df()

seg5_pivot = seg5.pivot_table(index=['seg1', 'seg2', 'seg3', 'seg4', 'seg5'], columns='crawl_year', values='n',
                              aggfunc='sum').fillna(0).astype(int)
seg5_pivot['total'] = seg5_pivot.sum(axis=1)
seg5_pivot = seg5_pivot.sort_values('total', ascending=False)
seg5_pivot

### File Extensions — transportation.gov

In [ ]:
# File extensions — ranked per year
ext_df = transportation_con.sql("""
    SELECT
        COALESCE(NULLIF(lower(regexp_extract(regexp_extract(surtkey, '\\)([^?]*)', 1), '\\.([a-zA-Z0-9]+)$', 1)), ''), '(none)') AS ext,
        crawl_year,
        COUNT(*) AS n
    FROM eot_captures
    GROUP BY 1, 2
""").df()

ext_pivot = ext_df.pivot(index='ext', columns='crawl_year', values='n').fillna(0).astype(int)
ext_pivot['total'] = ext_pivot.sum(axis=1)
ext_pivot = ext_pivot.sort_values('total', ascending=False)
ext_pivot

In [ ]:
ext_pivot.head(20)

### Filenames — transportation.gov

In [ ]:
# Filenames — ranked per year
fname_df = transportation_con.sql("""
    SELECT
        COALESCE(NULLIF(regexp_extract(regexp_extract(surtkey, '\\)([^?]*)', 1), '/([^/]+)$', 1), ''), '(root)') AS filename,
        crawl_year,
        COUNT(*) AS n
    FROM eot_captures
    GROUP BY 1, 2
""").df()

fname_pivot = fname_df.pivot(index='filename', columns='crawl_year', values='n').fillna(0).astype(int)
fname_pivot['total'] = fname_pivot.sum(axis=1)
fname_pivot = fname_pivot.sort_values('total', ascending=False)
fname_pivot

## Visualizations

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import numpy as np

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

In [ ]:
# Viz 1: Captures over time (bar chart, log scale)
year_counts = transportation_con.sql("""
    SELECT crawl_year, COUNT(*) AS captures
    FROM eot_captures
    GROUP BY 1 ORDER BY 1
""").df()

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(year_counts['crawl_year'], year_counts['captures'], color=sns.color_palette('Blues_d', 4))
ax.set_yscale('log')
ax.set_ylabel('Captures (log scale)')
ax.set_xlabel('Crawl Year')
ax.set_title('transportation.gov — Total Captures per Crawl Year')

for bar, val in zip(bars, year_counts['captures']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.15,
            f'{val:,}', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_ylim(1, year_counts['captures'].max() * 5)
plt.tight_layout()
plt.show()

In [ ]:
# Viz 2: Subdomain stacked bar — queries surthost_seg_2 directly (no helper, no regex)
host_year = transportation_con.sql("""
    SELECT
        COALESCE(surthost_seg_2, '(bare/www)') AS subdomain,
        crawl_year,
        COUNT(*) AS n
    FROM eot_captures
    WHERE surtkey IS NOT NULL
    GROUP BY 1, 2
""").df()

totals = host_year.groupby('subdomain')['n'].sum().sort_values(ascending=False)
top_subs = totals.head(8).index.tolist()
host_year['subdomain'] = host_year['subdomain'].where(
    host_year['subdomain'].isin(top_subs), 'other'
)

host_pivot = (host_year.pivot_table(index='crawl_year', columns='subdomain',
                                     values='n', aggfunc='sum')
                       .fillna(0).astype(int))
col_order = [s for s in top_subs if s in host_pivot.columns]
if 'other' in host_pivot.columns:
    col_order.append('other')
host_pivot = host_pivot[col_order]

ax = host_pivot.plot(kind='bar', stacked=True, figsize=(9, 6),
                     color=sns.color_palette('Set2', len(col_order)))
ax.set_ylabel('Captures')
ax.set_xlabel('Crawl Year')
ax.set_title('transportation.gov — Subdomain Breakdown per Crawl Year (surthost_seg_2)')
ax.legend(title='Subdomain', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
# Viz 3: Top 15 path segment 1 values — heatmap across crawl years
top15 = seg1_pivot[seg1_pivot.index != '(root)'].head(15)
year_cols = [c for c in top15.columns if c != 'total']

fig, ax = plt.subplots(figsize=(10, 8))
data = top15[year_cols].replace(0, np.nan)

sns.heatmap(np.log1p(top15[year_cols]), annot=top15[year_cols].values,
            fmt=',d', cmap='YlOrRd', linewidths=0.5, ax=ax,
            cbar_kws={'label': 'log(1 + count)'})
ax.set_title('transportation.gov — Top 15 SURT path_1 values by Crawl Year')
ax.set_ylabel('SURT path_1')
ax.set_xlabel('Crawl Year')
plt.tight_layout()
plt.show()

In [ ]:
# Viz 4: Seg1 churn — how many path segments are shared vs unique between 2016 and 2020
in_2016 = set(seg1_pivot[seg1_pivot['2016'] > 0].index)
in_2020 = set(seg1_pivot[seg1_pivot['2020'] > 0].index)

only_2016 = len(in_2016 - in_2020)
only_2020 = len(in_2020 - in_2016)
both = len(in_2016 & in_2020)

churn = pd.DataFrame({
    'category': ['Only in 2016', 'In both', 'Only in 2020'],
    'count': [only_2016, both, only_2020]
})

fig, ax = plt.subplots(figsize=(7, 5))
colors = ['#e74c3c', '#2ecc71', '#3498db']
bars = ax.bar(churn['category'], churn['count'], color=colors)
for bar, val in zip(bars, churn['count']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            str(val), ha='center', va='bottom', fontsize=13, fontweight='bold')
ax.set_ylabel('Number of unique seg1 values')
ax.set_title('transportation.gov — SURT path_1 Churn (2016 vs 2020)')
ax.set_ylim(0, max(churn['count']) * 1.15)
plt.tight_layout()
plt.show()

print(f"2016 had {len(in_2016)} unique seg1 values, 2020 had {len(in_2020)}")
print(f"{only_2016} disappeared, {only_2020} are new, {both} persisted")

In [ ]:
# Viz 5: Extension donut chart — top 10 + other
top_ext = ext_pivot.head(10).copy()
other_total = ext_pivot.iloc[10:]['total'].sum()
other_row = pd.DataFrame({'total': [other_total]}, index=['(other)'])
donut_data = pd.concat([top_ext[['total']], other_row])

fig, ax = plt.subplots(figsize=(8, 8))
colors = sns.color_palette('Set3', len(donut_data))
wedges, texts, autotexts = ax.pie(
    donut_data['total'], labels=donut_data.index, autopct='%1.1f%%',
    colors=colors, pctdistance=0.82, startangle=90
)
centre = plt.Circle((0, 0), 0.55, fc='white')
ax.add_patch(centre)
ax.set_title('transportation.gov — File Extension Distribution (all years)')
plt.tight_layout()
plt.show()

In [ ]:
# Viz 6: Extension shift — 2016 vs 2020 (excluding (none))
ext_no_none = ext_pivot[ext_pivot.index != '(none)'].head(10)
years = ['2016', '2020']

x = np.arange(len(ext_no_none))
width = 0.35

fig, ax = plt.subplots(figsize=(11, 6))
ax.bar(x - width/2, ext_no_none['2016'], width, label='2016', color='#3498db')
ax.bar(x + width/2, ext_no_none['2020'], width, label='2020', color='#e74c3c')

ax.set_xticks(x)
ax.set_xticklabels(ext_no_none.index, rotation=45, ha='right')
ax.set_ylabel('Captures')
ax.set_title('transportation.gov — Top 10 File Extensions: 2016 vs 2020')
ax.legend()
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.show()

In [ ]:
# Viz 7: Path hierarchy treemap (top 20 seg1 paths by total count)
try:
    import squarify
except ImportError:
    print("squarify not installed — run: pip install squarify")
    squarify = None

if squarify:
    tree_data = seg1_pivot[seg1_pivot.index != '(root)'].head(20).copy()
    labels = [f"{name}\n{total:,}" for name, total in zip(tree_data.index, tree_data['total'])]

    fig, ax = plt.subplots(figsize=(14, 8))
    colors = sns.color_palette('tab20', len(tree_data))
    squarify.plot(sizes=tree_data['total'], label=labels, color=colors, alpha=0.85,
                  text_kwargs={'fontsize': 9}, ax=ax)
    ax.set_title('transportation.gov — SURT path_1 Treemap (all years)', fontsize=14)
    ax.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
# Viz 8: PDF captures over time
pdf_row = ext_pivot.loc['pdf'] if 'pdf' in ext_pivot.index else None

if pdf_row is not None:
    year_cols = [c for c in ext_pivot.columns if c != 'total']
    pdf_by_year = pdf_row[year_cols]

    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar(pdf_by_year.index, pdf_by_year.values, color='#c0392b')
    for bar, val in zip(bars, pdf_by_year.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
                f'{val:,}', ha='center', va='bottom', fontsize=11, fontweight='bold')
    ax.set_ylabel('PDF Captures')
    ax.set_xlabel('Crawl Year')
    ax.set_title('transportation.gov — PDF File Captures per Crawl Year')
    ax.set_ylim(0, pdf_by_year.max() * 1.15)
    plt.tight_layout()
    plt.show()
else:
    print("No 'pdf' extension found in data.")

## Content Analysis

In [ ]:
# Analysis 1: Average URL depth per crawl year
depth_stats = transportation_con.sql("""
    SELECT
        crawl_year,
        COUNT(*) AS n,
        AVG(len(string_split(trim(regexp_extract(surtkey, '\\)([^?]*)', 1), '/'), '/'))) AS avg_depth,
        MEDIAN(len(string_split(trim(regexp_extract(surtkey, '\\)([^?]*)', 1), '/'), '/'))) AS median_depth,
        MAX(len(string_split(trim(regexp_extract(surtkey, '\\)([^?]*)', 1), '/'), '/'))) AS max_depth
    FROM eot_captures
    WHERE surtkey IS NOT NULL AND surtpath_1 IS NOT NULL AND surtpath_1 != ''
    GROUP BY 1 ORDER BY 1
""").df()

print("URL path depth statistics per crawl year:")
print(depth_stats.to_string(index=False))

# Bar chart of average depth
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(depth_stats['crawl_year'], depth_stats['avg_depth'], color=sns.color_palette('viridis', 4))
for bar, val in zip(bars, depth_stats['avg_depth']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{val:.2f}', ha='center', va='bottom', fontsize=11)
ax.set_ylabel('Average SURT path depth')
ax.set_xlabel('Crawl Year')
ax.set_title('transportation.gov — Average SURT path depth per Crawl Year')
plt.tight_layout()
plt.show()

In [ ]:
# Analysis 2: Clean URLs (no extension) vs static files (with extension) per year
clean_vs_static = transportation_con.sql("""
    SELECT
        crawl_year,
        SUM(CASE WHEN regexp_extract(regexp_extract(surtkey, '\\)([^?]*)', 1), '\\.([a-zA-Z0-9]+)$', 1) = '' THEN 1 ELSE 0 END) AS clean_urls,
        SUM(CASE WHEN regexp_extract(regexp_extract(surtkey, '\\)([^?]*)', 1), '\\.([a-zA-Z0-9]+)$', 1) != '' THEN 1 ELSE 0 END) AS static_files,
        COUNT(*) AS total
    FROM eot_captures
    GROUP BY 1 ORDER BY 1
""").df()

clean_vs_static['clean_pct'] = (clean_vs_static['clean_urls'] / clean_vs_static['total'] * 100).round(1)
print("Clean URLs (no extension) vs static files:")
print(clean_vs_static.to_string(index=False))

# Stacked percentage bar
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(clean_vs_static))
width = 0.6

ax.bar(x, clean_vs_static['clean_urls'], width, label='Clean URLs (no ext)', color='#2ecc71')
ax.bar(x, clean_vs_static['static_files'], width, bottom=clean_vs_static['clean_urls'],
       label='Static files (with ext)', color='#e67e22')

ax.set_xticks(x)
ax.set_xticklabels(clean_vs_static['crawl_year'])
ax.set_ylabel('Captures')
ax.set_xlabel('Crawl Year')
ax.set_title('transportation.gov — Clean URLs vs Static Files per Year')
ax.legend()
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.show()

In [ ]:
# Analysis 3: Top 30 PDF filenames
pdf_files = transportation_con.sql("""
    SELECT
        regexp_extract(regexp_extract(surtkey, '\\)([^?]*)', 1), '/([^/]+\\.pdf)$', 1) AS pdf_name,
        crawl_year,
        COUNT(*) AS n
    FROM eot_captures
    WHERE lower(regexp_extract(regexp_extract(surtkey, '\\)([^?]*)', 1), '\\.([a-zA-Z0-9]+)$', 1)) = 'pdf'
    GROUP BY 1, 2
""").df()

pdf_pivot = pdf_files.pivot_table(index='pdf_name', columns='crawl_year', values='n',
                                   aggfunc='sum').fillna(0).astype(int)
pdf_pivot['total'] = pdf_pivot.sum(axis=1)
pdf_pivot = pdf_pivot.sort_values('total', ascending=False)

print(f"Total unique PDF filenames: {len(pdf_pivot):,}")
print(f"\nTop 30 PDFs:")
pdf_pivot.head(30)

In [ ]:
# Analysis 4: Drupal node IDs — count and range per year
node_stats = transportation_con.sql("""
    SELECT
        crawl_year,
        COUNT(*) AS node_urls,
        MIN(TRY_CAST(surtpath_2 AS INTEGER)) AS min_node_id,
        MAX(TRY_CAST(surtpath_2 AS INTEGER)) AS max_node_id,
        COUNT(DISTINCT surtpath_2) AS unique_nodes
    FROM eot_captures
    WHERE surtpath_1 = 'node'
    GROUP BY 1 ORDER BY 1
""").df()

print("Drupal /node/ statistics per crawl year:")
print(node_stats.to_string(index=False))

# Bar chart
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(node_stats['crawl_year'], node_stats['unique_nodes'], color='#8e44ad')
for bar, val in zip(bars, node_stats['unique_nodes']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
            f'{val:,}', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_ylabel('Unique Drupal Node IDs')
ax.set_xlabel('Crawl Year')
ax.set_title('transportation.gov — Drupal Node IDs per Crawl Year')
plt.tight_layout()
plt.show()

### Export all tables as CSVs

In [ ]:
# Save all transportation.gov pivot tables as CSVs
# tables = {
#     'transportation_seg1': seg1_pivot,
#     'transportation_seg2': seg2_pivot,
#     'transportation_seg3': seg3_pivot,
#     'transportation_seg4': seg4_pivot,
#     'transportation_seg5': seg5_pivot,
#     'transportation_extensions': ext_pivot,
#     'transportation_filenames': fname_pivot,
# }

# for name, df in tables.items():
#     path = f'{name}.csv'
#     df.to_csv(path)
#     print(f'  {path} — {len(df):,} rows')

In [ ]:
transportation_con.close()

---
# Deduplicated URL Analysis — 7 Questions

Everything below uses **deduplicated URLs**: one row per unique SURT URL per crawl year
(no timestamps), with `dns:` records removed.

Source: the per-domain **cdxj** DuckDB (`eot_captures`), using the materialised SURT
columns (`surthost_seg_2` = subdomain 1, `surthost_seg_3` = subdomain 2). SURT
canonicalization already strips a leading `www` and lowercases the path.
Includes 2024 (cdxj has it; parquet does not yet).

Questions:
1. Number of subdomains over time
2. Separate directory path vs. filename-with-extension
3. Length of URL paths (paths only): segment presence, longest segments,
   hyphen/underscore usage, RFC 3986 reserved characters
4. Drupal node ID analysis
5. Common path words across segments/years
6. `/sites/` structure extraction
7. `.cfm` (ColdFusion) investigation

In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path

pd.set_option('display.max_rows', 120)
pd.set_option('display.max_colwidth', 100)
sns.set_theme(style='whitegrid')

DOMAIN = 'transportation.gov'
_q_candidates = [
    Path('data/12_transportation/cdxj.duckdb'),             # local layout (cwd = repo root)
    Path('../data/12_transportation/cdxj.duckdb'),          # local layout (cwd = domain_analysis/)
    Path('data/domains/12_transportation/cdxj.duckdb'),     # server layout (cwd = repo root)
    Path('../data/domains/12_transportation/cdxj.duckdb'),  # server layout (cwd = domain_analysis/)
]
QDB = next(p for p in _q_candidates if p.exists())
qcon = duckdb.connect(str(QDB), read_only=True)
print(f'Connected to {QDB}')

## Setup — deduplicate to unique URLs per year, remove DNS

One row per `(crawl_year, surtkey)`. `min()` picks a stable representative for the
display columns (they are functionally identical per surtkey anyway).

In [ ]:
qcon.execute(r'''
CREATE OR REPLACE TEMP TABLE dedup AS
SELECT crawl_year, surtkey,
       min(url)            AS url,
       min(surthost)       AS surthost,
       min(surthost_seg_2) AS sub1,   -- subdomain level 1 (e.g. 'data'); SURT strips 'www'
       min(surthost_seg_3) AS sub2,   -- subdomain level 2
       min(regexp_extract(surtkey, '\)([^?]*)', 1)) AS url_path  -- SURT path, query excluded
FROM eot_captures
WHERE url NOT LIKE 'dns:%'
GROUP BY 1, 2
''')

before = qcon.sql("SELECT crawl_year, COUNT(*) AS n FROM eot_captures GROUP BY 1 ORDER BY 1").df()
after  = qcon.sql("SELECT crawl_year, COUNT(*) AS n FROM dedup GROUP BY 1 ORDER BY 1").df()
dns    = qcon.sql("SELECT crawl_year, COUNT(*) AS removed_dns FROM eot_captures WHERE url LIKE 'dns:%' GROUP BY 1").df()
summary = before.merge(after, on='crawl_year', suffixes=('_raw', '_dedup'))
summary = summary.merge(dns, on='crawl_year', how='left').fillna({'removed_dns': 0})
summary['removed_dns'] = summary['removed_dns'].astype(int)
# re-captures: same surtkey fetched at multiple timestamps within the same crawl year
summary['removed_recaptures'] = summary['n_raw'] - summary['n_dedup'] - summary['removed_dns']
summary['redundancy_pct'] = (100 * summary['removed_recaptures'] / summary['n_raw']).round(1)
print(summary.to_string(index=False))
print('\nNote: DNS records exist only in the 2024 crawl; for earlier years the removed')
print('rows are entirely within-year re-captures of the same URL.')

## Q1 — Subdomains over time

Counts of unique URLs per subdomain per year, from the SURT host segments:
`surthost_seg_2` is the first subdomain label (e.g. `data` in `data.transportation.gov`),
`surthost_seg_3` the second. SURT already folds `www.transportation.gov` into the bare
domain, so NULL seg_2 means `(bare/www)`.

In [ ]:
# Q1a: unique-URL counts per subdomain-1 per year (pivot)
sub1_df = qcon.sql('''
    SELECT
        COALESCE(sub1, '(bare/www)') AS subdomain,
        crawl_year,
        COUNT(*) AS n
    FROM dedup
    GROUP BY 1, 2
''').df()

sub1_pivot = (sub1_df.pivot_table(index='subdomain', columns='crawl_year',
                                  values='n', aggfunc='sum')
                     .fillna(0).astype(int))
sub1_pivot['total'] = sub1_pivot.sum(axis=1)
sub1_pivot = sub1_pivot.sort_values('total', ascending=False)
print(f'Unique subdomain-1 labels: {len(sub1_pivot)}')
sub1_pivot

In [ ]:
# Q1b: subdomain-2 within subdomain-1 (e.g. foo.data.transportation.gov), where present
sub2_df = qcon.sql('''
    SELECT
        COALESCE(sub1, '(bare/www)') AS subdomain_1,
        sub2 AS subdomain_2,
        crawl_year,
        COUNT(*) AS n
    FROM dedup
    WHERE sub2 IS NOT NULL
    GROUP BY 1, 2, 3
''').df()
if len(sub2_df) > 0:
    sub2_pivot = (sub2_df.pivot_table(index=['subdomain_1', 'subdomain_2'],
                                      columns='crawl_year', values='n', aggfunc='sum')
                         .fillna(0).astype(int))
    sub2_pivot['total'] = sub2_pivot.sum(axis=1)
    display(sub2_pivot.sort_values('total', ascending=False).head(30))
else:
    print('No second-level subdomains found.')

In [ ]:
# Q1c: number of distinct subdomains per year (line chart)
subs_per_year = qcon.sql('''
    SELECT crawl_year,
           COUNT(DISTINCT COALESCE(sub1, '(bare/www)')) AS unique_subdomains
    FROM dedup
    GROUP BY 1 ORDER BY 1
''').df()

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(subs_per_year['crawl_year'], subs_per_year['unique_subdomains'],
        marker='o', linewidth=2)
for _, r in subs_per_year.iterrows():
    ax.annotate(str(r['unique_subdomains']), (r['crawl_year'], r['unique_subdomains']),
                textcoords='offset points', xytext=(0, 8), ha='center')
ax.set_xlabel('Crawl Year')
ax.set_ylabel('Distinct subdomains')
ax.set_title(f'{DOMAIN} — distinct subdomains per crawl year (deduplicated URLs)')
plt.tight_layout()
plt.show()

## Q2 — Directory path vs. filename with extension

A trailing segment matching `.<2-4 alphanumerics>` is treated as a **filename**;
everything before it is the **directory path**. Extension-less URLs (CMS clean URLs)
have no filename.

In [ ]:
qcon.execute(r'''
CREATE OR REPLACE TEMP TABLE paths AS
SELECT *,
    NULLIF(regexp_extract(url_path, '/([^/]+\.[a-zA-Z0-9]{2,4})$', 1), '') AS filename,
    NULLIF(lower(regexp_extract(url_path, '\.([a-zA-Z0-9]{2,4})$', 1)), '') AS extension,
    CASE WHEN regexp_matches(url_path, '/[^/]+\.[a-zA-Z0-9]{2,4}$')
         THEN regexp_replace(url_path, '/[^/]+\.[a-zA-Z0-9]{2,4}$', '')
         ELSE rtrim(url_path, '/')
    END AS dir_path
FROM dedup
''')

# URLs with vs without a filename, per year
qcon.sql('''
    SELECT crawl_year,
           COUNT(*) AS unique_urls,
           SUM(CASE WHEN filename IS NOT NULL THEN 1 ELSE 0 END) AS with_filename,
           SUM(CASE WHEN filename IS NULL THEN 1 ELSE 0 END) AS path_only,
           ROUND(100.0 * SUM(CASE WHEN filename IS NULL THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_path_only
    FROM paths
    GROUP BY 1 ORDER BY 1
''').show()

In [ ]:
# Extension table (filenames only — e.g. .jpg captured here, excluded from path analysis)
ext_pivot = qcon.sql('''
    SELECT extension, crawl_year, COUNT(*) AS n
    FROM paths
    WHERE extension IS NOT NULL
    GROUP BY 1, 2
''').df().pivot_table(index='extension', columns='crawl_year',
                      values='n', aggfunc='sum').fillna(0).astype(int)
ext_pivot['total'] = ext_pivot.sum(axis=1)
ext_pivot.sort_values('total', ascending=False).head(30)

## Q3 — Length of URL paths (directory paths only, filenames excluded)

`depth` = number of directory segments in `dir_path`. Then: segment presence,
longest segment per position, hyphen/underscore usage, and RFC 3986 reserved
characters.

In [ ]:
qcon.execute(r'''
CREATE OR REPLACE TEMP TABLE path_segs AS
SELECT crawl_year, surtkey, dir_path,
    CASE WHEN trim(dir_path, '/') = '' THEN 0
         ELSE len(string_split(trim(dir_path, '/'), '/')) END AS depth,
    list_extract(string_split(trim(dir_path, '/'), '/'), 1) AS seg1,
    list_extract(string_split(trim(dir_path, '/'), '/'), 2) AS seg2,
    list_extract(string_split(trim(dir_path, '/'), '/'), 3) AS seg3,
    list_extract(string_split(trim(dir_path, '/'), '/'), 4) AS seg4,
    list_extract(string_split(trim(dir_path, '/'), '/'), 5) AS seg5
FROM paths
''')

# Q3a: how many URLs have / don't have path_seg 1..5 (per year)
presence = qcon.sql('''
    SELECT crawl_year,
           COUNT(*) AS unique_urls,
           SUM(CASE WHEN depth >= 1 THEN 1 ELSE 0 END) AS has_seg1,
           SUM(CASE WHEN depth >= 2 THEN 1 ELSE 0 END) AS has_seg2,
           SUM(CASE WHEN depth >= 3 THEN 1 ELSE 0 END) AS has_seg3,
           SUM(CASE WHEN depth >= 4 THEN 1 ELSE 0 END) AS has_seg4,
           SUM(CASE WHEN depth >= 5 THEN 1 ELSE 0 END) AS has_seg5
    FROM path_segs
    GROUP BY 1 ORDER BY 1
''').df()
for i in range(1, 6):
    presence[f'lacks_seg{i}'] = presence['unique_urls'] - presence[f'has_seg{i}']
    presence[f'pct_has_seg{i}'] = (100 * presence[f'has_seg{i}'] / presence['unique_urls']).round(1)
presence[['crawl_year', 'unique_urls'] +
         [c for i in range(1, 6) for c in (f'has_seg{i}', f'lacks_seg{i}', f'pct_has_seg{i}')]]

In [ ]:
# Q3a (viz): depth distribution per year
depth_dist = qcon.sql('''
    SELECT crawl_year, LEAST(depth, 8) AS depth, COUNT(*) AS n
    FROM path_segs GROUP BY 1, 2
''').df().pivot_table(index='depth', columns='crawl_year', values='n', aggfunc='sum').fillna(0).astype(int)

ax = depth_dist.plot(kind='bar', figsize=(10, 5), width=0.8)
ax.set_xlabel('Directory depth (8 = 8+)')
ax.set_ylabel('Unique URLs')
ax.set_title(f'{DOMAIN} — directory path depth distribution per crawl year')
ax.set_yscale('log')
plt.tight_layout()
plt.show()

In [ ]:
# Q3b: longest segment value at each position
rows = []
for i in range(1, 6):
    r = qcon.sql(f'''
        SELECT seg{i}, len(seg{i}) AS length
        FROM path_segs
        WHERE seg{i} IS NOT NULL
        ORDER BY len(seg{i}) DESC
        LIMIT 1
    ''').fetchall()
    if r:
        rows.append({'position': i, 'longest_value': r[0][0], 'char_length': r[0][1]})
pd.DataFrame(rows)

In [ ]:
# Q3c: segments with the most hyphens / underscores (word separators)
qcon.sql(r'''
    WITH all_segs AS (
        SELECT unnest([seg1, seg2, seg3, seg4, seg5]) AS seg FROM path_segs
    ),
    seg_stats AS (
        SELECT seg,
               len(seg) - len(replace(seg, '-', '')) AS hyphens,
               len(seg) - len(replace(seg, '_', '')) AS underscores,
               COUNT(*) AS n
        FROM all_segs WHERE seg IS NOT NULL AND seg != ''
        GROUP BY seg
    )
    SELECT seg, hyphens, underscores, n
    FROM seg_stats
    ORDER BY greatest(hyphens, underscores) DESC
    LIMIT 20
''').df()

In [ ]:
# Q3d: RFC 3986 reserved characters in directory paths
# gen-delims: ? # [ ] @   (note: ? and # cannot appear in url_path — they delimit
# query/fragment upstream — but are included for completeness)
# sub-delims: ! $ & ' ( ) * + , ; =    plus % (percent-encoding)
reserved = ['?', '#', '[', ']', '@',
            '!', '$', '&', "'", '(', ')', '*', '+', ',', ';', '=', '%']
rows = []
for ch in reserved:
    n = qcon.execute(
        'SELECT COUNT(*) FROM path_segs WHERE contains(dir_path, ?)', [ch]
    ).fetchone()[0]
    rows.append({'char': ch, 'urls_containing': n})
res_df = pd.DataFrame(rows).sort_values('urls_containing', ascending=False)
total = qcon.sql('SELECT COUNT(*) FROM path_segs').fetchone()[0]
res_df['pct'] = (100 * res_df['urls_containing'] / total).round(3)
res_df

In [ ]:
# Q3d (samples): show example paths for the most common reserved characters
top_chars = res_df[res_df['urls_containing'] > 0].head(4)['char'].tolist()
for ch in top_chars:
    print(f'--- {ch!r} ---')
    samples = qcon.execute(
        'SELECT DISTINCT dir_path FROM path_segs WHERE contains(dir_path, ?) LIMIT 5', [ch]
    ).fetchall()
    for (s,) in samples:
        print('  ', s[:120])

## Q4 — Drupal node ID analysis

In [ ]:
node_stats = qcon.sql('''
    SELECT crawl_year,
           COUNT(*) AS node_urls,
           COUNT(DISTINCT seg2) AS unique_node_ids,
           MIN(TRY_CAST(seg2 AS INTEGER)) AS min_node_id,
           MAX(TRY_CAST(seg2 AS INTEGER)) AS max_node_id
    FROM path_segs
    WHERE seg1 = 'node'
    GROUP BY 1 ORDER BY 1
''').df()
if len(node_stats) > 0:
    print('Drupal /node/ statistics per crawl year (deduplicated URLs):')
    print(node_stats.to_string(index=False))
else:
    print('No /node/ paths found — domain may not use Drupal.')

## Q5 — Common path words across segments and years

Normalization: lowercase, split segments on any non-alphanumeric run
(hyphens, underscores, dots, %-encodings all become separators), drop pure numbers.
More aggressive NLP (stemming/lemmatisation) could fold plurals — noted as future work.

In [ ]:
word_df = qcon.sql(r'''
    WITH seg_pos AS (
        SELECT crawl_year, 1 AS pos, seg1 AS seg FROM path_segs WHERE seg1 IS NOT NULL
        UNION ALL SELECT crawl_year, 2, seg2 FROM path_segs WHERE seg2 IS NOT NULL
        UNION ALL SELECT crawl_year, 3, seg3 FROM path_segs WHERE seg3 IS NOT NULL
        UNION ALL SELECT crawl_year, 4, seg4 FROM path_segs WHERE seg4 IS NOT NULL
        UNION ALL SELECT crawl_year, 5, seg5 FROM path_segs WHERE seg5 IS NOT NULL
    ),
    words AS (
        SELECT crawl_year, pos,
               unnest(string_split(regexp_replace(lower(seg), '[^a-z0-9]+', ' ', 'g'), ' ')) AS word
        FROM seg_pos
    )
    SELECT word, pos, crawl_year, COUNT(*) AS n
    FROM words
    WHERE word != '' AND NOT regexp_matches(word, '^[0-9]+$')
    GROUP BY 1, 2, 3
''').df()

top10 = word_df.groupby('word')['n'].sum().nlargest(10)
print('Top 10 path words (all segments, all years):')
print(top10.to_string())

In [ ]:
# Word × segment-position pivot for the top 10 words
top_words = top10.index.tolist()
wp = (word_df[word_df['word'].isin(top_words)]
      .pivot_table(index='word', columns='pos', values='n', aggfunc='sum')
      .fillna(0).astype(int)
      .reindex(top_words))
wp.columns = [f'seg{c}' for c in wp.columns]
display(wp)

# Word × year pivot
wy = (word_df[word_df['word'].isin(top_words)]
      .pivot_table(index='word', columns='crawl_year', values='n', aggfunc='sum')
      .fillna(0).astype(int)
      .reindex(top_words))
display(wy)

## Q6 — `/sites/` structure extraction

Everything after `/sites/` — is there a common structure
(e.g. Drupal's `/sites/<site>/files/...`)?

In [ ]:
sites_df = qcon.sql(r'''
    SELECT
        regexp_extract(url_path, '/sites/([^/]+)', 1) AS sites_level1,
        regexp_extract(url_path, '/sites/[^/]+/([^/]+)', 1) AS sites_level2,
        crawl_year,
        COUNT(*) AS n
    FROM paths
    WHERE contains(url_path, '/sites/')
    GROUP BY 1, 2, 3
''').df()
if len(sites_df) > 0:
    sites_pivot = (sites_df.pivot_table(index=['sites_level1', 'sites_level2'],
                                        columns='crawl_year', values='n', aggfunc='sum')
                          .fillna(0).astype(int))
    sites_pivot['total'] = sites_pivot.sum(axis=1)
    display(sites_pivot.sort_values('total', ascending=False).head(30))
else:
    print('No /sites/ paths found.')

In [ ]:
# Deeper: full structure 3 levels below /sites/
qcon.sql(r'''
    SELECT
        regexp_extract(url_path, '(/sites/[^/]+/[^/]+/[^/]+)', 1) AS sites_prefix,
        COUNT(*) AS n
    FROM paths
    WHERE contains(url_path, '/sites/')
    GROUP BY 1
    ORDER BY n DESC
    LIMIT 20
''').df()

## Q7 — `.cfm` (ColdFusion) investigation

`.cfm` files indicate Adobe ColdFusion server pages — a legacy technology stack.

In [ ]:
cfm_by_year = qcon.sql('''
    SELECT crawl_year, COUNT(*) AS cfm_urls
    FROM paths WHERE extension = 'cfm'
    GROUP BY 1 ORDER BY 1
''').df()
if len(cfm_by_year) > 0:
    print('.cfm URLs per year (deduplicated):')
    print(cfm_by_year.to_string(index=False))
else:
    print('No .cfm URLs found.')

In [ ]:
# Top .cfm filenames and where they live
qcon.sql('''
    SELECT filename, dir_path, crawl_year, COUNT(*) AS n
    FROM paths
    WHERE extension = 'cfm'
    GROUP BY 1, 2, 3
    ORDER BY n DESC
    LIMIT 30
''').df()

In [ ]:
qcon.close()